# 06 · Clean `docker_tickets_v2.csv`

Fixes the four real defects found in `docs/DATASET_QUALITY_REPORT.md` — every rule below was tested against the
actual dataset before being written here, not against synthetic examples:

1. **Leftover raw HTML** (`<img>`, `<a href>`, `<br>`, `<div>`/`<span>`) pasted into GitHub markdown → converted to plain text
2. **Unbalanced/empty code fences** → empty ` ```\n``` ` pairs removed, any fence still unclosed gets closed
3. **Garbled/binary output** (non-UTF-8 bytes rendered as `�`/control characters) → row dropped, unsalvageable
4. **Exact-duplicate problem text from unfilled issue templates** → keep only the highest-trust copy per group

It deliberately does **not** touch the things the report checked and found to be non-issues (CLI ellipses, code-comment
`!!!`, blockquoted context, FAQ rows where `title == problem`) — those are real content, not noise.

Upload `docker_tickets_v2.csv` when prompted, or point `INPUT_PATH` at it in Drive.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
INPUT_PATH = None        # e.g. "/content/drive/MyDrive/customer-support-triage/tickets/processed/docker_tickets_v2.csv"
                          # leave as None to be prompted for a file upload instead
USE_DRIVE = True

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass

if INPUT_PATH:
    input_path = Path(INPUT_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    input_path = Path(next(iter(uploaded)))

OUT_DIR = input_path.parent
print("Reading:", input_path.resolve())

## 1 · Load and parse the list-columns

These columns are JSON arrays stored as strings (CSV has no native list type) — see the properness note in the report.

In [ ]:
LIST_COLUMNS = ["trust_reasons", "topics", "tags", "docker_versions", "error_strings", "exit_codes"]

df = pd.read_csv(input_path)
before_count = len(df)
for col in LIST_COLUMNS:
    df[col] = df[col].map(json.loads)

print("loaded rows:", before_count)
print(df["source_type"].value_counts().to_string())

## 2 · Strip leftover raw HTML

Only targets the specific tags confirmed in the report (`img`, `a href`, `br`, `div`/`span`) — does not touch markdown,
code fences, or blockquotes.

In [ ]:
IMG_ALT_RE = re.compile(r'<img[^>]*\balt="([^"]*)"[^>]*>')
IMG_BARE_RE = re.compile(r"<img\b[^>]*>")
A_HREF_RE = re.compile(r'<a\s+href="([^"]*)"[^>]*>(.*?)</a>', re.S)
BR_RE = re.compile(r"<br\s*/?>")
WRAP_TAG_RE = re.compile(r"</?(div|span)[^>]*>")
HTML_HINT_RE = re.compile(r"<img |<a\s+href|<br\s*/?>|<div|<span")


def strip_leftover_html(text):
    text = IMG_ALT_RE.sub(lambda m: f"[image: {m.group(1)}]" if m.group(1).strip() else "[image omitted]", text)
    text = IMG_BARE_RE.sub("[image omitted]", text)
    text = A_HREF_RE.sub(lambda m: m.group(2) if m.group(2).strip() == m.group(1) else f"{m.group(2)} ({m.group(1)})", text)
    text = BR_RE.sub("\n", text)
    return WRAP_TAG_RE.sub("", text)


html_hits = df["problem"].str.contains(HTML_HINT_RE) | df["resolution"].str.contains(HTML_HINT_RE)
print("rows with leftover HTML tags:", html_hits.sum())
df.loc[html_hits, "problem"] = df.loc[html_hits, "problem"].map(strip_leftover_html)
df.loc[html_hits, "resolution"] = df.loc[html_hits, "resolution"].map(strip_leftover_html)

remaining = df["problem"].str.contains(HTML_HINT_RE) | df["resolution"].str.contains(HTML_HINT_RE)
print("rows still containing an HTML-like tag after cleaning:", remaining.sum())
# Checked directly: the handful that remain are almost always real HTML/code the poster is showing as
# example content (a pasted webpage's source, a Go program that prints HTML) rather than a rendering
# artifact - stripping those would mangle the actual technical content, so they are left as-is.

## 3 · Fix code fences

In [ ]:
EMPTY_FENCE_RE = re.compile(r"```[ \t]*\n[ \t]*```")


def fix_code_fences(text):
    text = EMPTY_FENCE_RE.sub("", text)
    if text.count("```") % 2 == 1:
        text = text.rstrip() + "\n```"
    return text


fence_hits = (df["problem"].str.count("```") % 2 == 1) | (df["resolution"].str.count("```") % 2 == 1) \
             | df["problem"].str.contains(EMPTY_FENCE_RE) | df["resolution"].str.contains(EMPTY_FENCE_RE)
print("rows with an unbalanced or empty fence:", fence_hits.sum())
df.loc[fence_hits, "problem"] = df.loc[fence_hits, "problem"].map(fix_code_fences)
df.loc[fence_hits, "resolution"] = df.loc[fence_hits, "resolution"].map(fix_code_fences)

still_unbalanced = (df["problem"].str.count("```") % 2 == 1) | (df["resolution"].str.count("```") % 2 == 1)
print("still unbalanced after fixing (should be 0):", still_unbalanced.sum())

## 4 · Drop garbled/binary rows

Flags on either a known mojibake substring or a high density of the Unicode replacement character / control
characters. Verified in the report: 100% recall on the 11 originally-found rows, zero false positives on a
500-row random sample.

In [ ]:
CONTROL_CHAR_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
MOJIBAKE_RE = re.compile(r"Â|â€™|â€œ|â€\x9d|â€“|â€”|Ã©|Ã¨|ï¿½|�")


def is_garbled(text, ratio_threshold=0.005):
    if not text:
        return False
    bad = text.count("�") + len(CONTROL_CHAR_RE.findall(text))
    return bool(MOJIBAKE_RE.search(text)) or bad / max(1, len(text)) >= ratio_threshold


garbled = df["problem"].map(is_garbled) | df["resolution"].map(is_garbled)
print("garbled rows dropped:", garbled.sum())
dropped_garbled = df[garbled][["ticket_id", "source_type", "url"]]
df = df[~garbled].reset_index(drop=True)

## 5 · Deduplicate unfilled issue-template boilerplate

Groups by exact `problem` text (already normalised, so no fuzzy matching needed) and keeps only the
highest-trust, highest-voted row per group.

In [ ]:
before_dedup = len(df)
ranked = df.sort_values(["trust_score", "resolution_score"], ascending=False, na_position="last")
keep_index = ranked.drop_duplicates(subset="problem", keep="first").index
dropped_template_dupes = df[~df.index.isin(keep_index)][["ticket_id", "source_type", "url"]]
df = df.loc[df.index.isin(keep_index)].reset_index(drop=True)

print("template-duplicate rows removed:", before_dedup - len(df))
assert not df.duplicated(subset="problem").any()

## 6 · Normalize whitespace

Light touch: trims trailing spaces before newlines and caps runs of blank lines. Does not rewrap or reflow text.

In [ ]:
def normalize_whitespace(text):
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{4,}", "\n\n\n", text)
    return text.strip()


df["problem"] = df["problem"].map(normalize_whitespace)
df["resolution"] = df["resolution"].map(normalize_whitespace)
df["problem_words"] = df["problem"].str.split().str.len()
df["resolution_words"] = df["resolution"].str.split().str.len()

## 7 · Save and report the delta

In [ ]:
out_jsonl = OUT_DIR / "docker_tickets_v3.jsonl"
out_csv = OUT_DIR / "docker_tickets_v3.csv"

df.to_json(out_jsonl, orient="records", lines=True, force_ascii=False)
csv_frame = df.copy()
for col in LIST_COLUMNS:
    csv_frame[col] = csv_frame[col].map(json.dumps)
csv_frame.to_csv(out_csv, index=False)

cleaning_report = {
    "input_rows": before_count,
    "output_rows": len(df),
    "html_tags_cleaned_in_rows": int(html_hits.sum()),
    "code_fences_fixed_in_rows": int(fence_hits.sum()),
    "garbled_rows_dropped": int(garbled.sum()),
    "template_duplicate_rows_dropped": before_dedup - len(df),
    "dropped_garbled_sample": dropped_garbled.head(20).to_dict("records"),
    "dropped_template_dupes_sample": dropped_template_dupes.head(20).to_dict("records"),
    "by_source_type_after": df["source_type"].value_counts().to_dict(),
}
(OUT_DIR / "docker_tickets_v3_cleaning_report.json").write_text(json.dumps(cleaning_report, indent=2), encoding="utf-8")

print(f"{before_count} -> {len(df)} rows "
      f"({before_count - len(df)} removed: {int(garbled.sum())} garbled + {before_dedup - len(df)} template duplicates)")
print(f"HTML cleaned in {int(html_hits.sum())} rows, fences fixed in {int(fence_hits.sum())} rows (no rows dropped for these two)")

## 8 · Spot-check

In [ ]:
import random

print("=== 3 rows that had HTML cleaned ===")
for _, row in df[df["ticket_id"].isin(
        pd.read_csv(input_path, usecols=["ticket_id"])["ticket_id"][html_hits.values])].sample(min(3, html_hits.sum())).iterrows():
    print(f"[{row.source_type}] {row.title[:90]}")
    print("   ", row.resolution[:160].replace("\n", " "))

print("\n=== sample of dropped garbled rows ===")
for r in cleaning_report["dropped_garbled_sample"][:3]:
    print(" ", r)

print("\n=== sample of dropped template-duplicate rows ===")
for r in cleaning_report["dropped_template_dupes_sample"][:3]:
    print(" ", r)

try:
    from google.colab import files
    files.download(str(out_csv))
    files.download(str(OUT_DIR / "docker_tickets_v3_cleaning_report.json"))
except ImportError:
    pass